# AI-Assisted Triage System Evaluation Using A/B Testing

### Project Overview

Riverside General Hospital is a large urban teaching hospital evaluating whether an AI-assisted triage system can reduce Emergency Department (ED) Door-to-Doctor Time compared with the standard triage process.

Because real patient-level data cannot be used due to privacy and confidentiality requirements, this project uses a realistically simulated dataset designed to reflect Emergency Department operations over a three-month study period.

The dataset generated in this notebook will be used throughout the project for exploratory data analysis, statistical hypothesis testing, and executive reporting.

## Objectives

The objectives of this notebook are to:

- Generate a realistic Emergency Department dataset containing approximately 2,250 patient visits.
- Simulate patient demographics and clinical characteristics.
- Model Emergency Department operational workflows.
- Randomly assign patients to either the Control or Treatment group.
- Produce a clean dataset ready for validation and analysis.

In [ ]:
# Import libraries

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

In [ ]:
# Set random seed for reproducibility

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

In [ ]:
from datetime import datetime, timedelta

# ------------------------------------------
# Define the study period
# ------------------------------------------

study_start = datetime(2026, 1, 1)

study_days = 90

study_end = study_start + timedelta(days=study_days - 1)

print("Study Start:", study_start.date())
print("Study End:", study_end.date())
print("Study Duration:", study_days, "days")

In [ ]:
# ------------------------------------------
# Simulate daily patient arrivals
# ------------------------------------------

# Average number of patients arriving each day
AVERAGE_PATIENTS_PER_DAY = 25

# Generate the number of arrivals for each day
daily_arrivals = np.random.poisson(
    lam=AVERAGE_PATIENTS_PER_DAY,
    size=study_days
)

print("First 10 days:")
print(daily_arrivals[:10])

print("\nTotal simulated patients:", daily_arrivals.sum())

In [ ]:
# ------------------------------------------
# Define study size
# ------------------------------------------

# Target average number of patients per day
AVERAGE_PATIENTS_PER_DAY = 25

# Total number of simulated patients
NUM_PATIENTS = daily_arrivals.sum()

print(f"Total simulated patients: {NUM_PATIENTS}")

In [ ]:
# ------------------------------------------
# Generate unique Patient IDs
# ------------------------------------------

patient_ids = [
    f"P{str(i).zfill(6)}"
    for i in range(1, NUM_PATIENTS + 1)
]

patient_ids[:5]

In [ ]:
# ------------------------------------------
# Generate one visit date for each patient
# ------------------------------------------

# Create the list of study dates
study_dates = pd.date_range(start=study_start, periods=study_days)

# Repeat each date according to the number of arrivals that day
visit_dates = np.repeat(study_dates, daily_arrivals)

# Convert to a pandas Series for easier handling
visit_dates = pd.Series(visit_dates).dt.date

# Display the first few dates
visit_dates.head(10)

In [ ]:
print("Number of visit dates:", len(visit_dates))
print("Number of patient IDs:", len(patient_ids))

In [ ]:
# ------------------------------------------
# Define arrival hour probabilities
# ------------------------------------------

hours = np.arange(24)

hour_weights = np.array([
    1,  # 12 AM
    1,  # 1 AM
    1,  # 2 AM
    1,  # 3 AM
    1,  # 4 AM
    2,  # 5 AM
    3,  # 6 AM
    5,  # 7 AM
    7,  # 8 AM
    8,  # 9 AM
    9,  # 10 AM
    10, # 11 AM
    10, # 12 PM
    10, # 1 PM
    9,  # 2 PM
    9,  # 3 PM
    8,  # 4 PM
    7,  # 5 PM
    6,  # 6 PM
    5,  # 7 PM
    4,  # 8 PM
    3,  # 9 PM
    2,  # 10 PM
    1   # 11 PM
])

# Convert the weights into probabilities
hour_probabilities = hour_weights / hour_weights.sum()

In [ ]:
# ------------------------------------------
# Generate arrival hours
# ------------------------------------------

arrival_hours = np.random.choice(
    hours,
    size=NUM_PATIENTS,
    p=hour_probabilities
)

# Preview the first 10 arrival hours
arrival_hours[:10]

In [ ]:
# ------------------------------------------
# Generate arrival minutes
# ------------------------------------------

arrival_minutes = np.random.randint(
    0,
    60,
    size=NUM_PATIENTS
)

arrival_minutes[:10]

In [ ]:
# ------------------------------------------
# Combine hours and minutes into arrival time
# ------------------------------------------

arrival_times = [
    f"{hour:02d}:{minute:02d}"
    for hour, minute in zip(arrival_hours, arrival_minutes)
]

arrival_times[:10]

In [ ]:
# ------------------------------------------
# Generate patient ages
# ------------------------------------------

age_groups = [
    (0, 17),     # Children & Teenagers
    (18, 30),    # Young Adults
    (31, 50),    # Adults
    (51, 70),    # Older Adults
    (71, 90)     # Elderly
]

age_group_probabilities = [
    0.15,
    0.20,
    0.30,
    0.22,
    0.13
]

selected_groups = np.random.choice(
    len(age_groups),
    size=NUM_PATIENTS,
    p=age_group_probabilities
)

ages = [
    np.random.randint(
        age_groups[group][0],
        age_groups[group][1] + 1
    )
    for group in selected_groups
]

print("Minimum age:", min(ages))
print("Maximum age:", max(ages))

In [ ]:
# ------------------------------------------
# Create Age Group
# ------------------------------------------

def assign_age_group(age):

    if age <= 17:
        return "Child"

    elif age <= 30:
        return "Young Adult"

    elif age <= 50:
        return "Adult"

    elif age <= 70:
        return "Older Adult"

    else:
        return "Elderly"


age_category = [assign_age_group(age) for age in ages]

age_category[:10]

In [ ]:
# ------------------------------------------
# Generate patient sex
# ------------------------------------------

sex = np.random.choice(
    ["Male", "Female"],
    size=NUM_PATIENTS,
    p=[0.48, 0.52]
)

# Preview the first 10 values
sex[:10]

In [ ]:
# ------------------------------------------
# Check sex distribution
# ------------------------------------------

sex_distribution = pd.Series(sex).value_counts(normalize=True) * 100

print(sex_distribution.round(2))


In [ ]:
# ------------------------------------------
# Generate Chief Complaint
# ------------------------------------------

chief_complaints = [
    "Abdominal Pain",
    "Chest Pain",
    "Fever",
    "Headache",
    "Shortness of Breath",
    "Minor Injury",
    "Road Traffic Accident",
    "Skin Rash",
    "Urinary Tract Infection",
    "Burns",
    "Dizziness"
]

complaint_probabilities = [
    0.16,
    0.13,
    0.15,
    0.11,
    0.09,
    0.14,
    0.05,
    0.05,
    0.05,
    0.03,
    0.04
]

chief_complaint = np.random.choice(
    chief_complaints,
    size=NUM_PATIENTS,
    p=complaint_probabilities
)

chief_complaint[:10]

In [ ]:
print(sum(complaint_probabilities))

In [ ]:
# ------------------------------------------
# Triage level probabilities by chief complaint
# ------------------------------------------

triage_mapping = {
    "Chest Pain":               ([1, 2, 3], [0.10, 0.75, 0.15]),
    "Shortness of Breath":      ([1, 2, 3], [0.20, 0.70, 0.10]),
    "Road Traffic Accident":    ([1, 2, 3], [0.15, 0.65, 0.20]),
    "Abdominal Pain":           ([2, 3, 4], [0.20, 0.60, 0.20]),
    "Fever":                    ([2, 3, 4], [0.10, 0.70, 0.20]),
    "Headache":                 ([3, 4, 5], [0.20, 0.60, 0.20]),
    "Minor Injury":             ([3, 4, 5], [0.10, 0.50, 0.40]),
    "Skin Rash":                ([4, 5],    [0.30, 0.70]),
    "Urinary Tract Infection":  ([3, 4],    [0.60, 0.40]),
    "Burns":                    ([2, 3, 4], [0.30, 0.50, 0.20]),
    "Dizziness":                ([2, 3, 4], [0.20, 0.60, 0.20])
}

In [ ]:
# ------------------------------------------
# Assign triage level
# ------------------------------------------

triage_level = []

for complaint in chief_complaint:

    levels, probabilities = triage_mapping[complaint]

    assigned_level = np.random.choice(
        levels,
        p=probabilities
    )

    triage_level.append(assigned_level)

triage_level[:10]

In [ ]:
# ------------------------------------------
# Check triage distribution by complaint
# ------------------------------------------

triage_check = pd.crosstab(
    pd.Series(chief_complaint, name="Chief Complaint"),
    pd.Series(triage_level, name="Triage Level")
)

triage_check

In [ ]:
# ------------------------------------------
# Randomize patients into experiment groups
# ------------------------------------------

experiment_group = np.random.choice(
    ["Control", "AI"],
    size=NUM_PATIENTS,
    p=[0.5, 0.5]
)

experiment_group[:10]


In [ ]:
pd.Series(experiment_group).value_counts()

In [ ]:
# ------------------------------------------
# Generate Door-to-Triage Time (minutes)
# ------------------------------------------

door_to_triage = np.random.normal(
    loc=12,
    scale=4,
    size=NUM_PATIENTS
)

# Round and prevent negative values
door_to_triage = np.clip(
    np.round(door_to_triage),
    1,
    None
).astype(int)

door_to_triage[:10]

In [ ]:
# ------------------------------------------
# Generate Triage Duration
# ------------------------------------------

triage_duration = []

for group in experiment_group:

    if group == "Control":

        duration = np.random.normal(
            loc=8,
            scale=1.5
        )

    else:

        duration = np.random.normal(
            loc=5,
            scale=1.2
        )

    duration = max(2, round(duration))

    triage_duration.append(duration)

triage_duration[:10]

In [ ]:
# ------------------------------------------
# Generate Doctor Queue Time
# ------------------------------------------

doctor_queue_time = []

for level in triage_level:

    if level == 1:
        queue = np.random.normal(2, 1)

    elif level == 2:
        queue = np.random.normal(8, 3)

    elif level == 3:
        queue = np.random.normal(20, 6)

    elif level == 4:
        queue = np.random.normal(40, 10)

    else:
        queue = np.random.normal(65, 15)

    queue = max(0, round(queue))

    doctor_queue_time.append(queue)

doctor_queue_time[:10]

In [ ]:
# ------------------------------------------
# AI-assisted reduction in doctor queue time
# ------------------------------------------

adjusted_doctor_queue = []

for group, queue_time in zip(experiment_group, doctor_queue_time):

    if group == "AI":

        reduction = np.random.randint(3, 6)   # Reduce by 3–5 minutes
        adjusted_queue = max(0, queue_time - reduction)

    else:

        adjusted_queue = queue_time

    adjusted_doctor_queue.append(adjusted_queue)

In [ ]:
# ------------------------------------------
# Final Door-to-Doctor Time
# ------------------------------------------

door_to_doctor = (
    np.array(door_to_triage)
    + np.array(triage_duration)
    + np.array(adjusted_doctor_queue)
)

door_to_doctor[:10]

In [ ]:
# ------------------------------------------
# Create Final Emergency Department Dataset
# ------------------------------------------

ed_data = pd.DataFrame({
    "Patient_ID": patient_ids,
    "Visit_Date": visit_dates,
    "Arrival_Time": arrival_times,
    "Age": ages,
    "Age_Group": age_category,
    "Sex": sex,
    "Chief_Complaint": chief_complaint,
    "Triage_Level": triage_level,
    "Experiment_Group": experiment_group,
    "Door_to_Triage_Min": door_to_triage,
    "Triage_Duration_Min": triage_duration,
    "Doctor_Queue_Min": adjusted_doctor_queue,
    "Door_to_Doctor_Min": door_to_doctor
})

ed_data.head()

In [ ]:
# ------------------------------------------
# Data Validation
# ------------------------------------------

print("Dataset Shape:", ed_data.shape)

print("\nMissing Values")
print(ed_data.isnull().sum())

print("\nDuplicate Patient IDs")
print(ed_data["Patient_ID"].duplicated().sum())

print("\nData Types")
print(ed_data.dtypes)

In [ ]:
# ------------------------------------------
# Summary Statistics
# ------------------------------------------

ed_data.describe(include="all")

In [ ]:
# ------------------------------------------
# Export Dataset
# ------------------------------------------

ed_data.to_csv("Riverside_General_ED_AB_Test.csv", index=False)

print("Dataset exported successfully!")

In [ ]:
#from google.colab import files

# files.download("Riverside_General_ED_AB_Test.csv")

# Workstream 4: Exploratory Data Analysis (EDA)

## Objective

With the simulation dataset now complete, this workstream focuses on understanding the data before performing any statistical analysis.

The objectives are to:

- Verify the quality and completeness of the dataset.
- Understand the characteristics of the simulated Emergency Department population.
- Explore key operational metrics.
- Confirm that randomization produced comparable Control and AI groups.
- Identify any issues that should be addressed before hypothesis testing.

This exploratory analysis ensures that subsequent statistical conclusions are based on a well-understood and reliable dataset.

In [ ]:
# ------------------------------------------
# Display the first 10 patients
# ------------------------------------------

ed_data.head(10)

In [ ]:
# Number of rows and columns

ed_data.shape

In [ ]:
# Column names

ed_data.columns

In [ ]:
# summary of data types

ed_data.dtypes

# Workstream 4: Exploratory Data Analysis (EDA) and A/B Test Analysis

## Objective

The dataset has now been successfully generated through simulation. Before testing any hypotheses, it is important to understand the data and verify its quality.

This workstream focuses on:

- Understanding the Emergency Department population.
- Exploring the distribution of key variables.
- Validating that randomization produced comparable Control and AI groups.
- Performing statistical analysis to evaluate the impact of the AI-assisted triage system.
- Providing evidence-based recommendations to hospital leadership.

The goal is to ensure that every conclusion is supported by both statistical evidence and operational understanding.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
# Dataset dimensions

ed_data.shape

In [ ]:
ed_data.head()

In [ ]:
ed_data.dtypes

In [ ]:
ed_data.isnull().sum()

In [ ]:
ed_data.describe()

In [ ]:
# What does the age distribution of patients visiting Riverside General Hospital's Emergency Department look like?
# ------------------------------------------
# Age Distribution
# ------------------------------------------

fig = px.histogram(
    ed_data,
    x="Age",
    nbins=15,
    marginal="box",
    title="Distribution of Patient Age",
    labels={"Age": "Patient Age (Years)"},
)

fig.update_layout(
    xaxis_title="Patient Age (Years)",
    yaxis_title="Number of Patients",
    bargap=0.05
)

fig.show()

**Observation**: Most patients are between 20 and 29
years old.

**Interpretation**: The Emergency Department serves a predominantly adult population.

**Management Implication**: This suggests that staffing, resource planning, and clinical workflows should primarily support adult emergency care while still maintaining appropriate resources for pediatric and elderly patients.

In [ ]:
# At what time of day does Riverside General Hospital receive the highest volume of Emergency Department arrivals?

# ------------------------------------------
# ED Arrivals by Hour
# ------------------------------------------

arrival_counts = (
    ed_data["Arrival_Time"]
    .str[:2]
    .astype(int)
    .value_counts()
    .sort_index()
)

fig = px.bar(
    x=arrival_counts.index,
    y=arrival_counts.values,
    labels={
        "x": "Hour of Day",
        "y": "Number of Patients"
    },
    title="Emergency Department Arrivals by Hour"
)

fig.update_layout(
    xaxis=dict(dtick=1)
)

fig.show()

**Observation**: Patient arrivals peak at approximately 1:00 PM, making it the busiest hour of the day in Riverside General Hospital's Emergency Department.

**Interpretation**: The midday peak is operationally plausible. Patients may arrive following morning clinic referrals, workplace or road traffic incidents, and increasing walk-in visits as the day progresses. These factors combine to create higher Emergency Department demand around midday.

**Management implication:** Hospital leadership can use this information to optimize workforce scheduling and resource allocation. Ensuring adequate physician, nursing, and treatment room capacity during peak arrival hours may reduce patient congestion and improve Emergency Department throughput.

In [ ]:
# What are the most common chief complaints among patients visiting Riverside General Hospital's Emergency Department?

# ------------------------------------------
# Chief Complaint Distribution
# ------------------------------------------

complaint_counts = (
    ed_data["Chief_Complaint"]
    .value_counts()
    .reset_index()
)

complaint_counts.columns = ["Chief Complaint", "Number of Patients"]

fig = px.bar(
    complaint_counts,
    x="Chief Complaint",
    y="Number of Patients",
    title="Distribution of Chief Complaints in the Emergency Department",
    text="Number of Patients"
)

fig.update_layout(
    xaxis_title="Chief Complaint",
    yaxis_title="Number of Patients"
)

fig.show()

**Observation**: Abdominal pain and fever are the two most frequently reported chief complaints among Emergency Department visits.

**Interpretation**: The high frequency of abdominal pain and fever suggests that gastrointestinal and infectious presentations represent a substantial proportion of Emergency Department visits in our simulated hospital.

**Management implication:** Hospital leadership should ensure that the Emergency Department is adequately equipped and staffed to manage common gastrointestinal and infectious presentations. This includes maintaining appropriate clinical protocols, ensuring the availability of essential medications and diagnostic resources, and providing staff training for the most frequently encountered conditions.

In [ ]:
# How severe are the patients presenting to Riverside General Hospital's Emergency Department?

# ------------------------------------------
# Triage Level Distribution
# ------------------------------------------

triage_counts = (
    ed_data["Triage_Level"]
    .value_counts()
    .sort_index()
    .reset_index()
)

triage_counts.columns = ["Triage Level", "Number of Patients"]

fig = px.bar(
    triage_counts,
    x="Triage Level",
    y="Number of Patients",
    text="Number of Patients",
    title="Distribution of Emergency Department Triage Levels"
)

fig.update_layout(
    xaxis_title="Triage Level",
    yaxis_title="Number of Patients"
)

fig.show()

**Observation**: Most Emergency Department visits fall within the moderate-acuity triage categories, while relatively few patients present with life-threatening conditions requiring immediate intervention

**Interpretation**: This distribution aligns with our previous finding that abdominal pain and fever are the most common chief complaints. These conditions are frequently assigned moderate triage levels, which explains why the Emergency Department sees a larger proportion of moderate-acuity patients.

**Management implication:** Hospital leadership should align staffing levels, treatment room capacity, and clinical resources with the predominance of moderate-acuity patients, as these cases account for most Emergency Department activity. At the same time, the hospital must maintain adequate resuscitation facilities, specialized equipment, and trained personnel to respond rapidly to the smaller number of life-threatening and very urgent presentations

## Randomization Validation

Before evaluating the effectiveness of the AI-assisted triage system, it is important to verify that the Control and AI groups are comparable at baseline.

Random assignment is expected to produce groups with similar patient characteristics. If substantial differences exist before the intervention, any observed improvement may be attributable to baseline imbalances rather than the AI system itself.

This section evaluates whether key demographic and clinical characteristics are reasonably balanced between the two experimental groups.

In [ ]:
# ------------------------------------------
# Number of patients in each experimental group
# ------------------------------------------

group_counts = (
    ed_data["Experiment_Group"]
    .value_counts()
    .reset_index()
)

group_counts.columns = ["Group", "Number of Patients"]

fig = px.bar(
    group_counts,
    x="Group",
    y="Number of Patients",
    text="Number of Patients",
    title="Number of Patients by Experimental Group"
)

fig.update_traces(textposition="outside")

fig.show()

**Observation**:

The study population was almost evenly distributed between the experimental groups, with 1,128 patients assigned to the AI group and 1,125 patients assigned to the Control group.

**Interpretation:**

The difference of only three patients indicates that the randomization process produced two groups of nearly equal size, consistent with expectations for a randomized experiment.

**Management Implication:**

The balanced allocation of patients strengthens confidence that subsequent comparisons between the AI and Control groups are unlikely to be influenced by unequal group sizes. This supports a fair evaluation of the AI-assisted triage system.

In [ ]:
# Did randomization produce Control and AI groups with similar age distributions?

# ------------------------------------------
# Age Distribution by Experimental Group
# ------------------------------------------

fig = px.box(
    ed_data,
    x="Experiment_Group",
    y="Age",
    color="Experiment_Group",
    title="Patient Age by Experimental Group",
    labels={
        "Experiment_Group": "Experimental Group",
        "Age": "Patient Age (Years)"
    },
    points=False
)

fig.show()

In [ ]:
# Summary statistics by experimental group

ed_data.groupby("Experiment_Group")["Age"].describe().round(1)

In [ ]:
# Did randomization produce similar sex distributions in the AI and Control groups?

# ------------------------------------------
# Sex Distribution by Experimental Group
# ------------------------------------------

sex_distribution = pd.crosstab(
    ed_data["Experiment_Group"],
    ed_data["Sex"],
    margins=True
)

sex_distribution



In [ ]:
# Percentage distribution

sex_percent = pd.crosstab(
    ed_data["Experiment_Group"],
    ed_data["Sex"],
    normalize="index"
).round(3) * 100

sex_percent

In [ ]:
# ------------------------------------------
# Sex Distribution
# ------------------------------------------

sex_chart = (
    ed_data.groupby(["Experiment_Group", "Sex"])
    .size()
    .reset_index(name="Number of Patients")
)

fig = px.bar(
    sex_chart,
    x="Experiment_Group",
    y="Number of Patients",
    color="Sex",
    barmode="group",
    text="Number of Patients",
    title="Sex Distribution by Experimental Group"
)

fig.update_traces(textposition="outside")

fig.show()

In [ ]:
# Did randomization produce similar triage level distributions between the AI and Control groups?

# ------------------------------------------
# Triage Level Distribution by Experimental Group
# ------------------------------------------

triage_distribution = pd.crosstab(
    ed_data["Experiment_Group"],
    ed_data["Triage_Level"]
)

triage_distribution

In [ ]:
triage_percent = pd.crosstab(
    ed_data["Experiment_Group"],
    ed_data["Triage_Level"],
    normalize="index"
).round(3) * 100

triage_percent

In [ ]:
triage_chart = (
    ed_data.groupby(["Experiment_Group","Triage_Level"])
    .size()
    .reset_index(name="Patients")
)

fig = px.bar(
    triage_chart,
    x="Triage_Level",
    y="Patients",
    color="Experiment_Group",
    barmode="group",
    text="Patients",
    title="Triage Level Distribution by Experimental Group"
)

fig.update_layout(
    xaxis_title="Triage Level",
    yaxis_title="Number of Patients"
)

fig.show()

In [ ]:
# Did randomization produce similar chief complaint distributions between the AI and Control groups?

# ------------------------------------------
# Chief Complaint Distribution by Experimental Group
# ------------------------------------------

complaint_distribution = pd.crosstab(
    ed_data["Experiment_Group"],
    ed_data["Chief_Complaint"]
)

complaint_distribution

In [ ]:
complaint_percent = (
    pd.crosstab(
        ed_data["Experiment_Group"],
        ed_data["Chief_Complaint"],
        normalize="index"
    ) * 100
).round(1)

complaint_percent

In [ ]:
complaint_chart = (
    ed_data.groupby(["Experiment_Group", "Chief_Complaint"])
    .size()
    .reset_index(name="Patients")
)

fig = px.bar(
    complaint_chart,
    x="Experiment_Group",
    y="Patients",
    color="Chief_Complaint",
    title="Chief Complaint Distribution by Experimental Group",
    text="Patients"
)

fig.update_layout(
    xaxis_title="Experimental Group",
    yaxis_title="Number of Patients"
)

fig.show()

### Baseline Assessment Summary

The AI and Control groups were highly comparable across key demographic and clinical characteristics, including sample size, age, sex, triage level, and chief complaint.

No meaningful baseline imbalances were identified, suggesting that the randomization process was successful. Consequently, any subsequent differences in Door-to-Doctor time are more likely to reflect the effect of the AI-assisted triage system rather than pre-existing differences between the groups.

In [ ]:
# ------------------------------------------
# Average Door-to-Doctor Time
# ------------------------------------------

group_summary = (
    ed_data
    .groupby("Experiment_Group")["Door_to_Doctor_Min"]
    .agg(["count", "mean", "std", "median"])
    .round(2)
)

group_summary

In [ ]:
fig = px.box(
    ed_data,
    x="Experiment_Group",
    y="Door_to_Doctor_Min",
    color="Experiment_Group",
    points=False,
    title="Door-to-Doctor Time by Experimental Group",
    labels={
        "Experiment_Group": "Experimental Group",
        "Door_to_Doctor_Min": "Door-to-Doctor Time (minutes)"
    }
)

fig.show()

In [ ]:
from scipy.stats import ttest_ind

In [ ]:
# ------------------------------------------
# Independent Samples t-test
# ------------------------------------------

ai_group = ed_data.loc[
    ed_data["Experiment_Group"] == "AI",
    "Door_to_Doctor_Min"
]

control_group = ed_data.loc[
    ed_data["Experiment_Group"] == "Control",
    "Door_to_Doctor_Min"
]

t_statistic, p_value = ttest_ind(
    ai_group,
    control_group,
    equal_var=False
)

print(f"T-statistic: {t_statistic:.3f}")
print(f"P-value: {p_value:.20f}")

In [ ]:
# ------------------------------------------
# Calculate Cohen's d
# ------------------------------------------

import numpy as np

mean_ai = ai_group.mean()
mean_control = control_group.mean()

std_ai = ai_group.std()
std_control = control_group.std()

n_ai = len(ai_group)
n_control = len(control_group)

# Pooled Standard Deviation
pooled_std = np.sqrt(
    (
        ((n_ai - 1) * std_ai**2) +
        ((n_control - 1) * std_control**2)
    ) / (n_ai + n_control - 2)
)

cohens_d = (mean_ai - mean_control) / pooled_std

print(f"Cohen's d: {cohens_d:.3f}")

In [ ]:
import numpy as np
from scipy.stats import t

# Sample statistics
mean_ai = ai_group.mean()
mean_control = control_group.mean()

std_ai = ai_group.std()
std_control = control_group.std()

n_ai = len(ai_group)
n_control = len(control_group)

# Difference in means
mean_difference = mean_ai - mean_control

# Standard error (Welch's t-test)
se = np.sqrt((std_ai**2 / n_ai) + (std_control**2 / n_control))

# Welch-Satterthwaite degrees of freedom
df = (
    (std_ai**2 / n_ai + std_control**2 / n_control) ** 2
) / (
    ((std_ai**2 / n_ai) ** 2) / (n_ai - 1)
    + ((std_control**2 / n_control) ** 2) / (n_control - 1)
)

# Critical t value
t_critical = t.ppf(0.975, df)

# Confidence interval
lower = mean_difference - t_critical * se
upper = mean_difference + t_critical * se

print(f"Mean Difference (AI - Control): {mean_difference:.2f} minutes")
print(f"95% Confidence Interval: ({lower:.2f}, {upper:.2f})")